# Chapter 4 Marketing Analytics

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio
pio.renderers.default = "notebook"
import plotly.express as px


In [ ]:
mall0 = pd.read_csv('https://raw.githubusercontent.com/librowndev/Data-Science-for-Marketing-Analytics-Second-Edition/refs/heads/master/Chapter03/Datasets/Mall_Customers.csv')

In [ ]:
mall0.head()

In [ ]:
mall0.rename({'Annual Income (k$)': 'Income', 'Spending Score (1-100)': 'Spend_score'}, axis=1, inplace=True)

mall0.head()

In [ ]:
#mall0.plot.scatter(x="Income", y="Spend_score")
#plt.show()

#plt.scatter(data=mall0, x='Income', y='Spend_score')
#plt.show()

fig = px.scatter(data_frame=mall0, x="Income", y="Spend_score")
fig.show()


## Exercis 4.02 Chosing Clusters - Visual Inspection

In [ ]:
from sklearn.preprocessing import StandardScaler

cols_to_scale = ['Age', 'Income', 'Spend_score']

scaler = StandardScaler()

mall_scaled = mall0.copy()

mall_scaled[cols_to_scale] = scaler.fit_transform(mall_scaled[cols_to_scale])

In [ ]:
mall_scaled.describe()

from sklearn.cluster import KMeans

cluster_cols = ['Income', 'Spend_score']
markers = ['x', '*', '.', '|', '_', '+', '1']
plt.figure(figsize=[12, 8])

for n in range(2,8):
    model = KMeans(n_clusters=n, random_state=42)
    mall_scaled['Cluster']= model.fit_predict(mall_scaled[cluster_cols])
    plt.subplot(3,2, n-1)
    for clust in range (n):
        temp = mall_scaled[mall_scaled.Cluster == clust]
        plt.scatter(temp.Income, temp.Spend_score, marker=markers[clust], label='Cluster ' + str(clust), color='darkred')
        plt.title("N clustersL " + str(n))
        plt.xlabel('Income')
        plt.ylabel('Spend_score')
        plt.legend()
        
plt.show()

In [ ]:
from sklearn.cluster import KMeans
import plotly.graph_objects as go
from plotly.subplots import make_subplots

cluster_cols = ['Income', 'Spend_score']

# Plotly marker symbols (closest equivalents to x, *, ., |, _, +, 1)
symbols = ['x', 'star', 'circle', 'line-ns', 'line-ew', 'cross', 'y-up']

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"N clusters: {n}" for n in range(2, 8)],
)

for n in range(2, 8):
    model = KMeans(n_clusters=n, random_state=42)
    labels = model.fit_predict(mall_scaled[cluster_cols])
    row, col = divmod(n - 2, 2)

    for clust in range(n):
        mask = labels == clust
        fig.add_trace(
            go.Scatter(
                x=mall_scaled.loc[mask, 'Income'],
                y=mall_scaled.loc[mask, 'Spend_score'],
                mode='markers',
                marker=dict(
                    symbol=symbols[clust],
                    
                    size=7,
                    line=dict(width=1),  # needed for line-style symbols
                ),
                name=f'Cluster {clust}',
                legendgroup=f'n{n}',
                legendgrouptitle_text=f'N = {n}',
            ),
            row=row + 1, col=col + 1,
        )

fig.update_xaxes(title_text='Income')
fig.update_yaxes(title_text='Spend_score')
fig.update_layout(width=1100, height=900, title_text='KMeans clusters for N = 2 to 7')
fig.show()


### Exercise 4.03 Determine Number of Clusters - Elbow Method

#### Elbow Method

The elbow method plots the number of clusters vs. the **sum of squared errors (SSE)**. SKlearn has an intertia attribute available after fitting a model. The intertia value can be extracted like this: model.intertia_

By looping through a range of cluster values and extracting the resulting interia values to a list and then plotting in a line plot, the "elbow" can be visualy noted. The elbow is where the gain in terms of reduced erros slows down for additional clusters. It is a a simple and commonly used method for selecting the number of clusters.

In [ ]:
X = mall_scaled[cluster_cols]

inertia_scores = []

for K in range(2,11):
    inertia = KMeans(n_clusters=K, random_state=42).fit(X).inertia_
    inertia_scores.append(inertia)
    
#plt.figure(figsize=[7,5])
#plt.plot(range(2,11), inertia_scores, color='darkblue')
#plt.title("SSE/Inertia vs. number of clusters")
#plt.xlabel("Number of clusters")
#plt.ylabel("SSE/Inertia")
#plt.show()

fig = px.line(x=range(2,11),
              y=inertia_scores, 
              
             title='SSE/Inertia vs num of Clusters',
             labels={"x":'Number of Clusters',"y":"SSE/Inertia"})
fig.show()


# Option 1: inside px.line / px.scatter / etc.
fig = px.line(x=x_vals, y=y_vals, labels={"x": "Number of clusters", "y": "SSE/Inertia"})

# Option 2: after the figure exists (works for px and go)
fig.update_xaxes(title_text="Number of clusters")
fig.update_yaxes(title_text="SSE/Inertia")

# Option 3: via layout
fig.update_layout(xaxis_title="Number of clusters", yaxis_title="SSE/Inertia")


In [ ]:

import plotly.express as px

fig = px.line(
    x=list(range(2, 11)),
    y=inertia_scores,
    markers=True,
    title="SSE/Inertia vs. number of clusters",
    labels={"x": "Number of clusters", "y": "SSE/Inertia"},
)
fig.update_traces(line_color="darkblue")
fig.update_layout(width=1000, height=500)
fig.show()


### Exercise 4.04 Mean Shift Clustering

In [ ]:
#from sklearn.cluster import estimate_bandwidth
#bandwidth = estimate_bandwidth()

In [ ]:
from sklearn.cluster import MeanShift, estimate_bandwidth
bandwidth = 0.9

ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
ms.fit(mall_scaled[cluster_cols])
mall_scaled['Mean_cluster']= ms.predict(X)

In [ ]:
import plotly.graph_objects as go

# Plotly equivalents of x, *, ., |, _, +, 1
symbols = ['x', 'star', 'circle', 'line-ns', 'line-ew', 'cross', 'y-up']

fig = go.Figure()

for c in range(mall_scaled.Mean_cluster.nunique()):
    temp = mall_scaled[mall_scaled.Mean_cluster == c]
    fig.add_trace(
        go.Scatter(
            x=temp.Income,
            y=temp.Spend_score,
            mode='markers',
            marker=dict(
                symbol=symbols[c],
                
                size=8,
                line=dict(width=1, color='darkgreen'),  # needed for line-style symbols
            ),
            name=f'Cluster {c}',
        )
    )

fig.update_layout(
    xaxis_title='Income',
    yaxis_title='Spend_score',
    width=1100,
    height=600,
)
fig.show()


In [ ]:
from sklearn.cluster import KMeans
import plotly.graph_objects as go
from plotly.subplots import make_subplots

cluster_cols = ['Income', 'Spend_score']

# Plotly marker symbols (closest equivalents to x, *, ., |, _, +, 1)
symbols = ['x', 'star', 'circle', 'line-ns', 'line-ew', 'cross', 'y-up']

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"N clusters: {n}" for n in range(2, 8)],
)

for n in range(2, 8):
    model = KMeans(n_clusters=n, random_state=42)
    labels = model.fit_predict(mall_scaled[cluster_cols])
    row, col = divmod(n - 2, 2)

    for clust in range(n):
        mask = labels == clust
        fig.add_trace(
            go.Scatter(
                x=mall_scaled.loc[mask, 'Income'],
                y=mall_scaled.loc[mask, 'Spend_score'],
                mode='markers',
                marker=dict(
                    symbol=symbols[clust],
                    
                    size=7,
                    line=dict(width=1),  # needed for line-style symbols
                ),
                name=f'Cluster {clust}',
                legendgroup=f'n{n}',
                legendgrouptitle_text=f'N = {n}',
            ),
            row=row + 1, col=col + 1,
        )

fig.update_xaxes(title_text='Income')
fig.update_yaxes(title_text='Spend_score')
fig.update_layout(width=1100, height=900, title_text='KMeans clusters for N = 2 to 7')
fig.show()


In [ ]:
bandwidth = estimate_bandwidth(mall_scaled[cluster_cols], quantile=0.1)
print(bandwidth)

In [ ]:


ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
ms.fit(mall_scaled[cluster_cols])
mall_scaled['Mean_cluster']= ms.predict(mall_scaled[cluster_cols])
mall_scaled.Mean_cluster.nunique()


In [ ]:
%matplotlib inline
markers = ['x', '*', '.', '|', '_', '+', '1']

plt.figure(figsize=[8,6])

for c in range(mall_scaled.Mean_cluster.nunique()):
    temp = mall_scaled[mall_scaled.Mean_cluster ==c]
    plt.scatter(x=temp.Income, y=temp.Spend_score, marker=markers[c], label="Clusters " + str(c), color='darkgreen')
    
plt.xlabel("Income")
plt.ylabel("Spend_score")
plt.legend()
plt.show()

### Exercise 4.05 Clustering Data Using the K-Prototypes Method

In [ ]:
# read data from github (available in the project but for other devices without data)

bank0  = pd.read_csv('https://raw.githubusercontent.com/librowndev/Data-Science-for-Marketing-Analytics-Second-Edition/refs/heads/master/Chapter04/Datasets/Bank_Personal_Loan_Modelling-2.csv')

bank0.head()

In [ ]:
from sklearn.preprocessing import  StandardScaler

scaler = StandardScaler()

bank_scaled = bank0.copy()
bank_scaled['Income']= scaler.fit_transform((bank0[['Income']]))


In [ ]:
# import  kmodes
#
# cluster_cols = ['Income', 'Education']
#
# kp = KPrototypes(n_clusters = 3, random_state = 42)
#
# bank_scaled['Cluster ']= kp.fit_predict(bank_scaled[cluster_cols], categorical=[1])


### Exercise 4.06 Using Silhouette Score to Pick Optimal Number of Clusters

In [ ]:
# using data from exercise 4.02

cluster_cols = ['Income', 'Spend_score']

X = mall_scaled[cluster_cols]

model = KMeans(n_clusters=3, random_state=42)
cluster_assignments = model.fit_predict((X))

In [ ]:
from sklearn.metrics import silhouette_score

sil_avg = silhouette_score(X, cluster_assignments)

print(sil_avg)



In [ ]:
# loop over number of clusters then plot

sil_scores = []


for num in range(2, 11):
    model = KMeans(n_clusters=num, random_state=42)
    cluster_assignments = model.fit_predict(X)
    sil_avg = silhouette_score(X, cluster_assignments)
    sil_scores.append((sil_avg))



In [ ]:

plt.figure(figsize=[7,5])
plt.plot(range(2,11), sil_scores, color='blue')
plt.show()

In [ ]:

fig = px.line(
    x=list(range(2, 11)),
    y=sil_scores,
    markers=True,
    title="Sil Score",
    labels={"x": "Number of clusters", "y": "Avg Silhouette score"},
)
fig.update_traces(line_color="darkblue")
fig.update_layout(width=800, height=500)
fig.show()

### Exercise 4.07 Using Train-Test-Split to Evaluate Clustering Performance

The goal of the exercise it to ensure reliable and robust customer segments from the mall customers. The data will be separated into train and test sets first, the fit a K-Means model. If the clusters from the model are good the silhouette score should be consistent between the training and test data sets.

Train-test-split evaluates the performance of the clustering.

**Note**: Scaling of the data must be done after the train-test-split to avoid data leakage. The mean and standard deviation would "leak" because the test data would not be "unseen."

In [ ]:

from sklearn.model_selection import  train_test_split

train, test = train_test_split(mall0, train_size=0.75, random_state=42)

print(f"Shape of the training set: {train.shape}")
print(f"Shape of the test set  {test.shape}")

In [ ]:
model = KMeans(n_clusters=6, random_state=42)

train['Cluster'] = model.fit_predict(train[cluster_cols])

sil_avg = silhouette_score(train[cluster_cols], train['Cluster'])

print(sil_avg)


In [ ]:
test['Cluster'] = model.predict(test[cluster_cols])

score = silhouette_score(test[cluster_cols], test['Cluster'])

print(score)

In [ ]:
for c in range(test.Cluster.nunique()):
    temp = test[test.Cluster == c]
    plt.scatter(temp.Income, temp.Spend_score, marker=markers[c], color='blue')

plt.xlabel("Income")
plt.ylabel("Spend_score")
plt.title("the chart")
plt.show(())